[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/358121-vix-futures-trading-signals.ipynb)# Trading Signals In VIX Futures**M. Avellaneda, T. N. Li, A. Papanicolaou & G. Wang** — [arXiv:2103.02016](https://arxiv.org/abs/2103.02016) (November 2021)The paper models the VIX futures term structure as a mode-centred Gaussian VAR, simulates day-aheadscenarios, and each day takes whichever of five fixed (one-month, five-month) positions maximisesexpected utility. A five-layer, 550-unit PReLU network approximates that expected utility. Over tencross-validation folds it reports Sharpe ratios of **1.06 to 11.05, mean 4.53**, and fold profits of**35% to 725%**, before costs.This notebook rebuilds it from **public Cboe VIX futures settlements** — the same data VIX Centralrepublishes, which is where the paper says it got its curves — and prints **claimed → measured**.Three things it establishes, in order:1. The reconstruction is right. The roll weight, the position map and the settlements all match the   paper's own printed Table A.1 and its own data source, exactly.2. The network is approximating something with a **closed form**. Under the paper's own Gaussian VAR   the day-ahead P&L of every action is exactly Gaussian, so the expected utility can be written   down rather than learned — and the same rule solved that way needs no network at all.3. What the numbers do once the folds, the costs and the years after publication are handled honestly.Same data and logic as our engine, landing on the report's numbers. The full C0–C7 gate suite —deflation, PBO, crisis windows, spanning, capacity — lives in the report:**https://market-graphs.com/praxis/papers/358121**

In [ ]:
%pip install -q numpy==2.1.2 pandas==2.2.3 scipy==1.14.1 pyarrow==24.0.0 yfinance==0.2.65 matplotlib==3.9.2# torch is only needed for the optional retraining cell at the end.

## The libraryEverything the paper needs, rebuilt from its prose and its own printed tables. The formulas in the published PDF do not survive text extraction, so each function below records which equation it implements.

In [ ]:
# ── Praxis reproduction library — paper 358121 ────────────────────────────────# Self-contained: everything below rebuilds the paper's pipeline from the frozen# Cboe settlements. No imports beyond numpy/pandas/scipy (+ torch for the network).import io, json, hashlib, urllib.request, datetime as dtimport numpy as np, pandas as pdfrom scipy import linalg as sla, stats as sstREPO = "https://raw.githubusercontent.com/finique/praxis-reproductions/main/358121-vx-data"D, N_STATE, PPY = 5, 11, 252STATE_COLS = ["vix"] + [f"V{i}" for i in range(1, D + 1)] + [f"R{i}" for i in range(1, D + 1)]ACTIONS = np.array([(0, 0), (-1, 1), (-1, 2), (1, -1), (1, -2)], float)      # eq (3.1)ACTION_LABELS = ["(0,0)", "(-1,1)", "(-1,2)", "(1,-1)", "(1,-2)"]GAMMA = {"pwl": 1.3, "exp": 3.0}                                             # (3.5), (3.6)HALF_SPREAD = 0.025                                                          # $/contract/sideFOLDS = [("2008-04-16","2009-07-17"),("2009-07-20","2010-10-19"),("2010-10-20","2012-01-23"),         ("2012-01-24","2013-04-29"),("2013-04-30","2014-07-30"),("2014-07-31","2015-10-29"),         ("2015-10-30","2017-02-01"),("2017-02-02","2018-05-04"),("2018-05-07","2019-08-07"),         ("2019-08-08","2020-11-05")]                                        # Table 3.1PAPER_END, SAMPLE_END = "2020-11-06", "2026-08-31"# ── data ──────────────────────────────────────────────────────────────────────def _dl(name):    with urllib.request.urlopen(f"{REPO}/{name}", timeout=120) as r:        return r.read()def load_frozen():    """Cboe VX daily settlements (Settle = DCRP), monthly contracts, columns keyed by    contract expiry; plus the Cboe VIX index. Frozen 2026-09-07 so this notebook lands    on the published numbers; set REFETCH=True to pull live from Cboe and diff."""    man = json.loads(_dl("MANIFEST.json"))    out = {}    for f in ("vx_settlements.parquet", "vix_index.parquet"):        b = _dl(f)        assert hashlib.sha256(b).hexdigest() == man[f]["sha256"], f"{f}: checksum mismatch"        out[f] = pd.read_parquet(io.BytesIO(b))    panel = out["vx_settlements.parquet"]    panel.columns = pd.to_datetime(panel.columns)    return panel, out["vix_index.parquet"]["close"]def load_curve(panel, vix, start="2008-04-14", end=SAMPLE_END, n=6):    """Daily F1..F6 (the six nearest monthly contracts) + the roll weight w.    w = (T1 - t)/(T1 - T0) in CALENDAR days, T0 = the expiry that just passed. This is    the schedule stated at the top of the paper's Section 3 ("100% in the front-month as    soon as the prior future matures, then 0% at the next maturity date") and it    reproduces the `w` column of the paper's Table A.1 to five decimals."""    exp = np.array([e.date() for e in panel.columns])    recs = []    for t in panel.index:        idx = np.where(exp > t.date())[0]        if len(idx) < n:            continue        idx = idx[:n]        vals = panel.iloc[panel.index.get_loc(t), idx].to_numpy(float)        if not np.isfinite(vals).all():            continue        t1 = exp[idx[0]]        t0 = exp[idx[0] - 1] if idx[0] > 0 else t1 - dt.timedelta(days=30)        cyc = (t1 - t0).days        if cyc <= 0:            continue        r = {"date": t, "w": min(max((t1 - t.date()).days / cyc, 0.0), 1.0), "cyc": cyc}        for k in range(n):            r[f"F{k+1}"] = vals[k]            r[f"E{k+1}"] = pd.Timestamp(exp[idx[k]])        recs.append(r)    c = pd.DataFrame(recs).set_index("date").sort_index()    c["nd"] = np.r_[np.diff(c.index.values).astype("timedelta64[D]").astype(int), 1]    c["vix"] = vix.reindex(c.index)    return c.loc[start:end].dropna(subset=[f"F{k+1}" for k in range(n)] + ["vix"])def build_state(c):    """X_t = (VIX, V^1..V^5, Roll^1..Roll^5) — the paper's eleven input neurons.    V^i = w F^i + (1-w) F^{i+1} is the constant-maturity future (eq 2.2);    Roll^i = -(F^{i+1} - F^i)/(cyc V^i) is the roll yield per calendar day (eq 2.4-2.5),    negative in contango, which is why the rolling strategies decay."""    w, cyc = c["w"].to_numpy(float), c["cyc"].to_numpy(float)    col = {"vix": c["vix"].to_numpy(float)}    for i in range(1, D + 1):        fi, fj = c[f"F{i}"].to_numpy(float), c[f"F{i+1}"].to_numpy(float)        v = w * fi + (1 - w) * fj        col[f"V{i}"], col[f"R{i}"] = v, -(fj - fi) / (cyc * v)    return pd.DataFrame({k: col[k] for k in STATE_COLS}, index=c.index)def realised_returns(c, panel):    """Daily returns of the one-month and five-month rolling strategies, marked on the    PHYSICAL contracts held (eq 2.3). Row t carries the return earned from t to t+1, so    it is what the action chosen at t receives — no look-ahead."""    st, idx = build_state(c), c.index    w = c["w"].to_numpy(float)    out = {}    for h in (1, 5):        v = st[f"V{h}"].to_numpy(float)        near, far = c[f"E{h}"].to_numpy(), c[f"E{h+1}"].to_numpy()        pn, pf = c[f"F{h}"].to_numpy(float), c[f"F{h+1}"].to_numpy(float)        r = np.full(len(idx), np.nan)        for k in range(len(idx) - 1):            try:                a, b = panel.at[idx[k + 1], near[k]], panel.at[idx[k + 1], far[k]]            except KeyError:                continue            if np.isfinite(a) and np.isfinite(b):                r[k] = (w[k] * (a - pn[k]) + (1 - w[k]) * (b - pf[k])) / v[k]        out[f"I{h}"] = r    return pd.DataFrame(out, index=idx)# ── the model ─────────────────────────────────────────────────────────────────def modal_state(X, grid=512):    """X* — the paper centres the state on the MODE, not the mean, "for robustness    because CMFs have a heavy right skew". Figure 2.1 plots a modal curve, i.e. one    modal value per coordinate, so we take per-coordinate KDE modes."""    return np.array([np.linspace(x.min(), x.max(), grid)[        int(np.argmax(sst.gaussian_kde(x)(np.linspace(x.min(), x.max(), grid))))]        for x in X.T])class VAR1:    """psi_t = X_t - X*,  psi_{t+1} = A psi_t + Z,  Z ~ N(0, Sigma). Least squares, no    intercept (eq 2.6). `pairs[t]` marks (t, t+1) as usable — False at fold seams, the    paper's "exclude the pasting outlier"."""    def __init__(self, A, S, mode):        self.A, self.Sigma, self.mode = A, S, mode        self.chol = np.linalg.cholesky(S + 1e-14 * np.eye(len(S)))        self.stat_cov = sla.solve_discrete_lyapunov(A, S)        self.stat_chol = np.linalg.cholesky(            self.stat_cov + 1e-12 * np.eye(len(S)) * np.trace(self.stat_cov) / len(S))        self.max_eig = float(np.abs(np.linalg.eigvals(A)).max())    @classmethod    def fit(cls, X, pairs, mode=None):        mode = modal_state(X) if mode is None else mode        psi = X - mode        p0, p1 = psi[:-1][pairs[:-1]], psi[1:][pairs[:-1]]        A = np.linalg.solve(p0.T @ p0, p0.T @ p1).T        E = p1 - p0 @ A.T        return cls(A, E.T @ E / (len(E) - 1), mode)    def sample_stationary(self, n, rng):        return self.mode + rng.standard_normal((n, len(self.mode))) @ self.stat_chol.T    def step_mean(self, X):        return self.mode + (X - self.mode) @ self.A.T    def step(self, X, m, rng):        return (self.step_mean(X)[:, None, :]                + rng.standard_normal((X.shape[0], m, len(self.mode))) @ self.chol.T)def _coeffs(X, nd):    """R^h_{t+1} = c_h . X_{t+1} + k_h, exactly. This affinity is the whole point: it    makes R(a) | X_t Gaussian under the paper's own VAR."""    n = X.shape[0]    nd = np.full(n, float(nd)) if np.isscalar(nd) else np.asarray(nd, float)    C, K = np.zeros((n, 2, N_STATE)), np.full((n, 2), -1.0)    for j, h in enumerate((1, 5)):        C[:, j, h] = 1.0 / X[:, h]        C[:, j, D + h] = nd    return C, Kdef action_moments(var, X, nd=1.0):    """Conditional mean and sd of R_{t+1}(a) for all five actions — exact."""    C, K = _coeffs(X, nd)    m_h = np.einsum("nhj,nj->nh", C, var.step_mean(X)) + K    cov = np.einsum("nhk,ngk->nhg", np.einsum("nhj,jk->nhk", C, var.Sigma), C)    return m_h @ ACTIONS.T, np.sqrt(np.maximum(        np.einsum("ph,nhg,pg->np", ACTIONS, cov, ACTIONS), 0.0))def utility(r, kind, g):    return np.where(r >= 0, r, g * r) if kind == "pwl" else (1 - np.exp(-g * r)) / gdef expected_utility_closed_form(mu, sd, kind, g):    """E[U(R(a))|X_t] in closed form — the quantity the 5x550 network approximates.      pwl : mu - (g-1)(sd phi(z) - mu Phi(-z)),  z = mu/sd      exp : (1 - exp(-g mu + g^2 sd^2 / 2)) / g   [argmax == argmax of mu - g sd^2/2]"""    if kind == "pwl":        z = np.divide(mu, sd, out=np.zeros_like(mu), where=sd > 0)        neg = np.where(sd > 0, sd * sst.norm.pdf(z) - mu * sst.norm.cdf(-z), np.maximum(-mu, 0))        return mu - (g - 1.0) * neg    return (1 - np.exp(-g * mu + 0.5 * g ** 2 * sd ** 2)) / gdef expected_utility_mc(var, X, kind, g, m, rng, nd=1.0, chunk=4000):    """The paper's own estimator: M one-step draws per state, average the utility."""    C, K = _coeffs(X, nd)    out = np.empty((X.shape[0], 5))    for s in range(0, X.shape[0], chunk):        e = min(s + chunk, X.shape[0])        rh = np.einsum("nhj,nmj->nmh", C[s:e], var.step(X[s:e], m, rng)) + K[s:e][:, None, :]        out[s:e] = utility(np.einsum("nmh,ph->nmp", rh, ACTIONS), kind, g).mean(1)    return out# ── trading ───────────────────────────────────────────────────────────────────def contract_counts(actions, c, wealth):    """Appendix A (A.1): n_i = a * P * weight / (1000 * F_i). Columns (n1, n2, n5, n6).    Reproduces every position printed in the paper's Table A.1."""    a, w = ACTIONS[actions], c["w"].to_numpy(float)    f = {i: c[f"F{i}"].to_numpy(float) for i in (1, 2, 5, 6)}    return np.column_stack([a[:, 0] * wealth * w / (1000 * f[1]),                            a[:, 0] * wealth * (1 - w) / (1000 * f[2]),                            a[:, 1] * wealth * w / (1000 * f[5]),                            a[:, 1] * wealth * (1 - w) / (1000 * f[6])])def backtest(actions, c, rets, half_spread=0.0, bps=0.0, p0=100_000.0):    """Costs per the paper's (3.8)-(3.9): half the bid-ask spread ($0.025/contract, tick    $0.05) plus `bps` of contract value (1000 x F), so the dollar cost widens by itself    when the curve is in backwardation. Position changes are netted on the PHYSICAL    contract, so the monthly roll pays its real cost. Result is independent of p0."""    n = len(c)    a, w = ACTIONS[actions], c["w"].to_numpy(float)    gross = a[:, 0] * rets["I1"].to_numpy(float) + a[:, 1] * rets["I5"].to_numpy(float)    ecol = {k: c[f"E{k}"].to_numpy() for k in range(1, 7)}    px = {k: c[f"F{k}"].to_numpy(float) for k in range(1, 7)}    p, net, cf, prev = np.empty(n + 1), np.full(n, np.nan), np.zeros(n), {}    p[0] = p0    for t in range(n):        want = {}        for leg, (kn, kf) in ((0, (1, 2)), (1, (5, 6))):            for k, wt in ((kn, w[t]), (kf, 1 - w[t])):                q = a[t, leg] * p[t] * wt / (1000 * px[k][t])                if q:                    want[ecol[k][t]] = want.get(ecol[k][t], 0.0) + q        # a contract keeps trading after the front rolls off, it just changes slot --        # search all six, and only fall back to F1 once it leaves the window entirely        price = {ecol[k][t]: px[k][t] for k in range(1, 7)}        cost = 0.0        for e in set(want) | set(prev):            dq = abs(want.get(e, 0.0) - prev.get(e, 0.0))            if dq:                f = price.get(e, px[1][t])                cost += dq * (half_spread * 1000 + bps / 1e4 * 1000 * f)        cf[t] = cost / p[t]        net[t] = (gross[t] if np.isfinite(gross[t]) else 0.0) - cf[t]        p[t + 1] = p[t] * (1 + net[t])        prev = want    net[~np.isfinite(gross)] = np.nan    return pd.Series(net, index=c.index).dropna()def metrics(r, rf=0.0):    """The paper's footnote 5: annualised excess return [prod(1+R)]^(252/T) - (1+rf)    over annualised std R x sqrt(252). `sharpe_mean_std` is the conventional    mean/std x sqrt(252) alongside, because the two differ materially at these levels."""    x = r.dropna().to_numpy(float)    T = len(x)    g = float(np.prod(1 + x))    ann = g ** (PPY / T) - (1 + rf) if g > 0 else np.nan    sd = float(x.std(ddof=1) * np.sqrt(PPY))    lvl = np.r_[1.0, np.cumprod(1 + x)]    return {"E_R": ann, "std_R": sd, "profit_pct": (g - 1) * 100,            "sharpe": ann / sd if sd > 0 else np.nan,            "sharpe_mean_std": float(x.mean() / x.std(ddof=1) * np.sqrt(PPY)),            "mdd": float((lvl / np.maximum.accumulate(lvl) - 1).min()), "n": T}def fold_masks(idx):    return [((idx >= a) & (idx <= b)) for a, b in FOLDS]def contig(rows):    p = np.zeros(len(rows), bool)    p[:-1] = rows[:-1] & rows[1:]    return p# ── what the paper printed ────────────────────────────────────────────────────# Table 3.2, piece-wise-linear utility, no transaction costs:#   (E[R_t], std[R_t], Profit %, Sharpe, Maximum Drawdown) per fold.CLAIMED = [(2.361, 0.443, 304.264, 5.297, -0.196), (1.195, 0.368, 145.924, 3.215, -0.138),           (4.951, 0.447, 724.848, 11.053, -0.117), (2.835, 0.410, 384.387, 6.878, -0.214),           (0.854, 0.242, 108.168, 3.474, -0.128), (1.129, 0.361, 137.044, 3.093, -0.123),           (1.027, 0.375, 121.582, 2.709, -0.156), (1.415, 0.754, 130.578, 1.862, -0.293),           (0.329, 0.302, 34.784, 1.056, -0.180), (3.284, 0.491, 429.191, 6.661, -0.240)]# Table 3.4, same signal with transaction costs: fold -> {c bps: (profit %, Sharpe, MDD)}CLAIMED_COST = {    0: {0: (202.813, 3.756, -0.213), 20: (187.321, 3.509, -0.225), 30: (138.855, 2.709, -0.244), 40: (85.275, 1.787, -0.269)},    1: {0: (35.406, 0.956, -0.216), 20: (24.631, 0.719, -0.262), 30: (-28.295, -0.488, -0.473), 40: (-84.254, -1.677, -0.857)},    2: {0: (391.423, 6.643, -0.163), 20: (370.000, 6.329, -0.163), 30: (247.105, 4.460, -0.187), 40: (103.636, 2.096, -0.377)},    3: {0: (152.749, 3.099, -0.258), 20: (148.617, 3.025, -0.262), 30: (87.970, 1.919, -0.307), 40: (-7.369, 0.046, -0.510)},    4: {0: (-6.926, -0.159, -0.329), 20: (-6.926, -0.159, -0.329), 30: (-11.656, -0.324, -0.346), 40: (-43.848, -1.458, -0.515)},    5: {0: (8.136, 0.345, -0.311), 20: (8.126, 0.344, -0.311), 30: (-1.441, 0.119, -0.333), 40: (-42.591, -0.898, -0.507)},    6: {0: (-14.775, -0.175, -0.306), 20: (-14.780, -0.175, -0.306), 30: (-31.083, -0.561, -0.367), 40: (-76.575, -1.658, -0.765)},    7: {0: (-26.051, -0.049, -0.601), 20: (-26.122, -0.050, -0.601), 30: (-28.788, -0.087, -0.601), 40: (-57.851, -0.509, -0.681)},    8: {0: (-52.547, -1.403, -0.542), 20: (-52.550, -1.403, -0.542), 30: (-55.441, -1.492, -0.569), 40: (-80.190, -2.170, -0.802)},    9: {0: (218.103, 3.768, -0.240), 20: (195.975, 3.439, -0.240), 30: (123.111, 2.315, -0.279), 40: (11.942, 0.456, -0.528)},}

## 1 · Is the reconstruction right?Before any performance number, three checks against things the paper itself prints.

In [ ]:
# ── 1 · Rebuild the curve, and check it against the paper's own numbers ───────panel, vix = load_frozen()curve = load_curve(panel, vix)state = build_state(curve)rets  = realised_returns(curve, panel)print(f"curve: {len(curve)} trading days, {curve.index.min().date()} -> {curve.index.max().date()}")print(f"       {len(curve.loc[:'2020-11-06'])} days inside the paper's own window "      f"(it reports ~3,165)\n")# (a) The roll weight w. The paper prints it to five decimals in Table A.1.print("Table A.1 roll weight w:")for day, want in [("2020-12-28", 0.65714), ("2020-12-29", 0.62857),                  ("2020-12-30", 0.60000), ("2020-12-31", 0.57143)]:    got = float(curve.at[pd.Timestamp(day), "w"])    print(f"   {day}  ours {got:.5f}   paper {want:.5f}   {'MATCH' if abs(got-want)<5e-6 else 'DIFF'}")# (b) The position map of Appendix A, against the contracts Table A.1 prints.print("\nTable A.1 positions  (n1, n2, n5, n6) and the net position:")for day, p_idx, act, want, net in [("2020-12-28", 100.00, 2, (-3, -1, 5, 3), 4),                                   ("2020-12-29", 101.25, 1, (-3, -1, 2, 1), -1)]:    n = contract_counts(np.array([act]), curve.loc[[pd.Timestamp(day)]],                        np.array([p_idx * 1000.0]))[0]    got = tuple(int(x) for x in np.round(n))    print(f"   {day}  a={ACTION_LABELS[act]:>7}  ours {str(got):>18}  paper {str(want):>18}  "          f"net {sum(got):>3} vs {net:>3}   {'MATCH' if got == want and sum(got) == net else 'DIFF'}")# (c) The settlements themselves, against VIX Central -- the source the paper names.VIX_CENTRAL = {"2019-11-04": [14.725, 16.225, 17.325, 17.975, 18.075, 18.275],               "2019-11-05": [15.025, 16.575, 17.625, 18.225, 18.325, 18.525],               "2019-11-06": [14.775, 16.325, 17.575, 18.225, 18.375, 18.575],               "2019-11-07": [14.625, 16.175, 17.425, 18.125, 18.275, 18.525],               "2019-11-08": [14.275, 16.025, 17.325, 18.075, 18.225, 18.500],               "2019-11-11": [14.075, 15.825, 17.125, 17.875, 18.075, 18.375],               "2019-11-12": [13.875, 15.825, 17.175, 17.925, 18.175, 18.425]}bad = sum(abs(float(curve.at[pd.Timestamp(d), f"F{i+1}"]) - v[i]) > 1e-9          for d, v in VIX_CENTRAL.items() for i in range(6))print(f"\nVIX Central cross-check, 7 days x 6 contracts: {42 - bad}/42 exact")# (d) The two rolling strategies the whole paper trades. Both must decay in contango.r = rets.loc[:"2020-11-06"].dropna()for h, name in ((1, "one-month (I1)"), (5, "five-month (I5)")):    m = metrics(r[f"I{h}"])    print(f"   {name:16} {m['E_R']*100:7.1f}%/yr   vol {m['std_R']*100:5.1f}%   "          f"cumulative {m['profit_pct']:7.1f}%")

## 2 · The model — and the closed formThe network approximates `E[U(R(a))|X]` for five actions. But `R(a)` is affine in tomorrow's state and tomorrow's state is Gaussian by assumption, so that expectation is an exact formula.

In [ ]:
# ── 2 · The model, and the closed form the network is approximating ──────────# The state is 11 numbers: VIX, five constant-maturity futures, five roll yields.# The paper fits a mode-centred Gaussian VAR(1) to it, simulates day-ahead scenarios,# and picks whichever of five fixed positions has the highest expected utility.idx = curve.indexX = state[STATE_COLS].to_numpy(float)masks = fold_masks(idx)train = np.zeros(len(idx), bool)for g, mk in enumerate(masks):    if g != 9:        train |= mkvar = VAR1.fit(X, contig(train), mode=modal_state(X[train]))print("the modal curve X* (the paper: front future 12-14%, long end 17-20%, upward sloping,")print("roll yields negative):")print("   VIX %.1f  |  CMF %s  |  roll %%/day %s" % (    var.mode[0], np.round(var.mode[1:6], 1), np.round(var.mode[6:] * 100, 3)))print(f"   VAR max|eigenvalue| {var.max_eig:.4f} (stationary)\n")# R_{t+1}(a) is AFFINE in X_{t+1}, and X_{t+1}|X_t is Gaussian by assumption.# So the expected utility the 5x550 network is trained to approximate has a closed form.Xe = X[masks[9]]for util in ("pwl", "exp"):    mu, sd = action_moments(var, Xe, 1.0)    cf = expected_utility_closed_form(mu, sd, util, GAMMA[util])    mc300 = expected_utility_mc(var, Xe, util, GAMMA[util], 300, np.random.default_rng(0))    mc20k = expected_utility_mc(var, Xe, util, GAMMA[util], 20000, np.random.default_rng(1))    print(f"{util}: closed form vs Monte Carlo  corr {np.corrcoef(cf.ravel(), mc20k.ravel())[0,1]:.6f} "          f"(M=20,000)   max|diff| {np.abs(cf - mc20k).max():.2e}")    print(f"      same action chosen: {100*(cf.argmax(1)==mc20k.argmax(1)).mean():.1f}% at M=20,000, "          f"{100*(cf.argmax(1)==mc300.argmax(1)).mean():.1f}% at the paper's M=300")

## 3 · Claimed → measuredThe paper's own ten folds, train on nine, test on the tenth. `restated` is the paper's own fold Sharpe recomputed from its own profit and its own volatility at the true fold length — see the report.

In [ ]:
# ── 3 · Claimed -> measured, the paper's own 10-fold design ──────────────────# The network is a function approximator for a stated rule, so we run the rule two ways:#   PROXY   the same argmax solved exactly, no network at all  (deterministic here)#   NETWORK the paper's 5x550 PReLU net (its published actions; cell 5 retrains one)acts_pub = pd.read_parquet(io.BytesIO(_dl("actions_seed0.parquet")))proxy_a = np.zeros(len(idx), int)for f, test in enumerate(masks):    tr = np.zeros(len(idx), bool)    for g, mk in enumerate(masks):        if g != f:            tr |= mk    v = VAR1.fit(X, contig(tr), mode=modal_state(X[tr]))    mu, sd = action_moments(v, X, 1.0)    proxy_a[test] = expected_utility_closed_form(mu, sd, "pwl", GAMMA["pwl"]).argmax(1)[test]dfn_a = acts_pub["kfold_pwl"].to_numpy()print(f"{'fold':>4} {'window':>23} | {'claimed':>8} {'restated':>8} | {'network':>8} {'proxy':>7} | "      f"{'claimed %':>10} {'network %':>10}")rows = []for f, mk in enumerate(masks):    mn, mp = metrics(backtest(dfn_a[mk], curve[mk], rets[mk])), metrics(backtest(proxy_a[mk], curve[mk], rets[mk]))    er, sd_, pr, sr, mdd = CLAIMED[f]    restated = ((1 + pr / 100) ** (252 / 316) - 1) / sd_    rows.append((sr, restated, mn["sharpe"], mp["sharpe"], pr, mn["profit_pct"]))    print(f"{f:>4} {FOLDS[f][0]}..{FOLDS[f][1]} | {sr:8.2f} {restated:8.2f} | "          f"{mn['sharpe']:8.2f} {mp['sharpe']:7.2f} | {pr:10.1f} {mn['profit_pct']:10.1f}")a = np.array(rows)print(f"{'MEAN':>4} {'':>23} | {a[:,0].mean():8.2f} {a[:,1].mean():8.2f} | "      f"{a[:,2].mean():8.2f} {a[:,3].mean():7.2f} | {a[:,4].mean():10.1f} {a[:,5].mean():10.1f}")full = (idx >= FOLDS[0][0]) & (idx <= FOLDS[9][1])pooled = metrics(backtest(dfn_a[full], curve[full], rets[full]))print(f"\nPooled across the whole 2008-2020 out-of-fold series (the honest aggregate, "      f"not an average of ten 15-month Sharpes):")print(f"   Sharpe {pooled['sharpe']:.2f}   {pooled['E_R']*100:.1f}%/yr   vol {pooled['std_R']*100:.1f}%   "      f"max drawdown {pooled['mdd']*100:.1f}%")

## 4 · BenchmarksThe four constant actions and SPY are the paper's Appendix B. The roll-sign rules are the one-line version of what the paper says its signal does.

In [ ]:
# ── 4 · The benchmarks the paper prints, and the one it does not ─────────────import yfinance as yfspy = yf.download("SPY", start="2008-01-01", end="2026-09-05", progress=False,                  auto_adjust=True)["Close"]spy = (spy.iloc[:, 0] if isinstance(spy, pd.DataFrame) else spy)spy.index = pd.to_datetime(spy.index).tz_localize(None)spy_r = spy.pct_change().reindex(idx)contango = state["R1"].to_numpy(float) < 0bench = {f"constant {ACTION_LABELS[j]}": np.full(len(idx), j) for j in range(1, 5)}bench["roll-sign: contango long 1m/short 5m"] = np.where(contango, 3, 1)bench["roll-sign: contango short 1m/long 5m"] = np.where(contango, 1, 3)print(f"{'benchmark':>42} | {'mean fold SR':>12} {'folds +':>8} | {'pooled SR':>10} {'ann %':>8} {'MDD %':>8}")for nm, aa in bench.items():    fs = [metrics(backtest(aa[mk], curve[mk], rets[mk]))["sharpe"] for mk in masks]    p = metrics(backtest(aa[full], curve[full], rets[full]))    print(f"{nm:>42} | {np.mean(fs):12.2f} {sum(x>0 for x in fs):>6}/10 | "          f"{p['sharpe']:10.2f} {p['E_R']*100:8.1f} {p['mdd']*100:8.1f}")fs = [metrics(spy_r[mk].dropna())["sharpe"] for mk in masks]p = metrics(spy_r[full].dropna())print(f"{'SPY (the paper Table B.1)':>42} | {np.mean(fs):12.2f} {sum(x>0 for x in fs):>6}/10 | "      f"{p['sharpe']:10.2f} {p['E_R']*100:8.1f} {p['mdd']*100:8.1f}")fs = [metrics(backtest(dfn_a[mk], curve[mk], rets[mk]))["sharpe"] for mk in masks]print(f"{'>> the paper strategy (network)':>42} | {np.mean(fs):12.2f} {sum(x>0 for x in fs):>6}/10 | "      f"{pooled['sharpe']:10.2f} {pooled['E_R']*100:8.1f} {pooled['mdd']*100:8.1f}")

## 5 · Costs, and the years after publication

In [ ]:
# ── 5 · The cost ladder, and what is left after 2020 ─────────────────────────# Section 3.3: half the bid-ask spread ($0.025 per contract, tick $0.05) plus c bps of# contract value, c in {20, 30, 40}.print(f"{'':>18} " + "".join(f"{'c=%dbps' % c:>22}" for c in (0, 20, 30, 40)))print(f"{'fold':>18} " + "".join(f"{'ours':>10}{'paper':>12}" for _ in range(4)))for f, mk in enumerate(masks):    line = f"{f:>18} "    for c in (0, 20, 30, 40):        m = metrics(backtest(dfn_a[mk], curve[mk], rets[mk], half_spread=HALF_SPREAD, bps=c))        line += f"{m['sharpe']:10.2f}{CLAIMED_COST[f][c][1]:12.2f}"    print(line)print(f"\n{'pooled 2008-2020':>18} " + "".join(    f"{metrics(backtest(dfn_a[full], curve[full], rets[full], half_spread=HALF_SPREAD, bps=c))['sharpe']:10.2f}"    f"{'':>12}" for c in (0, 20, 30, 40)))post = (idx > PAPER_END) & (idx <= SAMPLE_END)print(f"\nPost-publication, model trained once through {PAPER_END} and never refitted:")for nm, mk in (("2008-2020 (in sample)", full), ("2021-2026 (out of sample)", post)):    g = metrics(backtest(acts_pub['decay_pwl'].to_numpy()[mk], curve[mk], rets[mk]))    n = metrics(backtest(acts_pub['decay_pwl'].to_numpy()[mk], curve[mk], rets[mk],                         half_spread=HALF_SPREAD, bps=30))    print(f"   {nm:26} n={g['n']:>5}   gross Sharpe {g['sharpe']:6.2f} ({g['E_R']*100:7.1f}%/yr)   "          f"at 30bps {n['sharpe']:6.2f} ({n['E_R']*100:7.1f}%/yr)")

## 6 · Retrain the network yourself

In [ ]:
# ── 6 · Retrain the network from scratch (optional, slow on a free Colab CPU) ─# The network only approximates E[U(R(a))|X] for the five actions. Retraining it is the# honest check that nothing above depends on our particular fit -- but it is also where# the paper's result is least reproducible: the initialisation, the 100,000 stationary# draws and the 300 one-step draws per state are all random, and no seed is published.RETRAIN = False        # set True to run (~5 min per fold on a Colab CPU)if RETRAIN:    import torch    from torch import nn    def train_fold(var, seed):        rng = np.random.default_rng(seed)        raw = var.sample_stationary(135_000, rng)        keep = raw[(raw[:, 0] > 5) & (raw[:, 1:6] > 5).all(1)][:100_000]        Y = expected_utility_mc(var, keep, "pwl", GAMMA["pwl"], 300, rng)        mu, sd = keep.mean(0), np.where(keep.std(0) > 0, keep.std(0), 1.0)        ys = float(np.abs(Y).mean())        torch.manual_seed(seed)        mods, prev = [], 11        for _ in range(5):                                   # 5 hidden layers            mods += [nn.Linear(prev, 550), nn.PReLU(550, init=0.1)]; prev = 550        net = nn.Sequential(*mods, nn.Linear(550, 5), nn.PReLU(5, init=0.1))        opt, lf = torch.optim.Adam(net.parameters(), 1e-3), nn.MSELoss()        xt = torch.tensor((keep - mu) / sd, dtype=torch.float32)        yt = torch.tensor(Y / ys, dtype=torch.float32)        g = torch.Generator().manual_seed(seed)        for _ in range(15):                                  # 15 epochs, batch 160            for s in range(0, len(xt), 160):                i = torch.randperm(len(xt), generator=g)[s:s + 160]                opt.zero_grad(); lf(net(xt[i]), yt[i]).backward(); opt.step()        with torch.no_grad():            q = net(torch.tensor((X - mu) / sd, dtype=torch.float32)).numpy()        return q.argmax(1)    fresh = np.zeros(len(idx), int)    for f, test in enumerate(masks):        tr = np.zeros(len(idx), bool)        for g_, mk in enumerate(masks):            if g_ != f:                tr |= mk        v = VAR1.fit(X, contig(tr), mode=modal_state(X[tr]))        fresh[test] = train_fold(v, 100 * f)[test]        print(f"  fold {f} retrained, Sharpe "              f"{metrics(backtest(fresh[mk], curve[mk], rets[mk]))['sharpe']:.2f}")    print(f"\nfresh fit agrees with the published actions on "          f"{100*(fresh[full]==dfn_a[full]).mean():.1f}% of days")    print(f"pooled Sharpe: fresh {metrics(backtest(fresh[full], curve[full], rets[full]))['sharpe']:.2f} "          f"vs published {pooled['sharpe']:.2f}")else:    print("RETRAIN = False. Everything above is exact without it: the proxy, the")    print("benchmarks, the cost ladder and the decay run are all deterministic, and the")    print("network's own actions are the ones behind the published report.")

## 7 · The picture

In [ ]:
# ── 7 · The picture ─────────────────────────────────────────────────────────import matplotlib.pyplot as plteq   = (1 + backtest(dfn_a[full], curve[full], rets[full])).cumprod()eqp  = (1 + backtest(proxy_a[full], curve[full], rets[full])).cumprod()eqb  = (1 + backtest(np.full(int(full.sum()), 2), curve[full], rets[full])).cumprod()eqc  = (1 + backtest(dfn_a[full], curve[full], rets[full],                     half_spread=HALF_SPREAD, bps=30)).cumprod()fig, ax = plt.subplots(2, 1, figsize=(11, 8), height_ratios=[3, 2], sharex=True)ax[0].plot(eq.index, eq, lw=1.4, label="network, no costs (the paper's basis)")ax[0].plot(eqp.index, eqp, lw=1.1, label="same rule, closed form, no network")ax[0].plot(eqc.index, eqc, lw=1.1, label="network, half-spread + 30bps")ax[0].plot(eqb.index, eqb, lw=1.0, ls="--", c="0.5", label="constant (-1,2), no model")ax[0].set_yscale("log"); ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)ax[0].set_title("Paper 358121 - out-of-fold equity, 2008-2020 (log scale)")dd = eq / eq.cummax() - 1ax[1].fill_between(dd.index, dd * 100, 0, alpha=.5)ax[1].set_ylabel("drawdown %"); ax[1].grid(alpha=.3)plt.tight_layout(); plt.show()